<a href="https://colab.research.google.com/github/lilyb838/Final-project/blob/main/Projectv5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.1 MB/s eta 0:00:00


In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz
from collections import defaultdict
from itertools import combinations
import networkx as nx
from collections import Counter
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = pd.read_csv('/content/drive/MyDrive/raceform.csv')

/tmp/ipykernel_662/3453427977.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/raceform.csv')


In [6]:
print(df.shape)

(1851285, 37)


In [7]:
(df.isna().mean() * 100).round(4)

,0
date,0.0000
course,0.0000
race_id,0.0000
off,0.0000
race_name,0.0000
type,0.0000
class,41.5141
pattern,85.7743
rating_band,58.4214
age_band,0.0086


In [9]:
def convert_sp(x):
    if pd.isna(x) or x == '':
        return None

    x = x.replace('F', '').replace('J', '').replace('C', '')

    if x == 'Evens' or x == "Evs":
        return 1.0

    if '/' in x:
        num, den = x.split('/')
        return float(num) / float(den)

    try:
        return float(x)
    except:
        return None

df['sp'] = df['sp'].apply(convert_sp)

In [10]:
df['ovr_btn'] = pd.to_numeric(
    df['ovr_btn'].replace(['-'], np.nan),
    errors='coerce'
)

In [11]:
df = df.rename(columns={"class": "race_class"})
df = df.rename(columns={"type": "race_type"})

In [12]:
#convert distance to furlongs
def convert_dist(x):
  if pd.isna(x):
        return None

  x = str(x).strip()

  miles = re.search(r'(\d+)m', x)
  furlongs = re.search(r'(\d+(?:/\d+)?)f', x)

  total = 0

  if miles:
      total += int(miles.group(1)) * 8

  if furlongs:
      f = furlongs.group(1)
      if '/' in f:
          num, den = f.split('/')
          total += float(num) / float(den)
      else:
          total += float(f)

  return total

df['dist'] = df['dist'].apply(convert_dist)

In [13]:
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

In [14]:
cols_to_drop = list(df.columns[df.isna().mean() > 0.5]) +["comment", "num", "rpr", "ts", "prize", "time", "ran", "race_name", "btn" ]
df = df.drop(columns=cols_to_drop)

In [15]:
#fill blanks with 'MISSING'
df['sp'] = df['sp'].fillna(df['sp'].median())
df['draw'] = df['draw'].fillna('MISSING')
df['draw'] = df['draw'].astype(str)
df['race_class'] = df['race_class'].fillna('MISSING')
df['draw'] = df['draw'].fillna('MISSING')
df['age_band'] = df['age_band'].fillna('MISSING')
df['going'] = df['going'].fillna('MISSING')
df['jockey'] = df['jockey'].fillna('MISSING')
df['trainer'] = df['trainer'].fillna('MISSING')
df['draw'] = df['draw'].fillna('MISSING')
df['dam'] = df['dam'].fillna('MISSING')
df['damsire'] = df['damsire'].fillna('MISSING')
df['owner'] = df['owner'].fillna('MISSING')

In [16]:
df['pos_numeric'] = pd.to_numeric(df['pos'], errors='coerce').fillna(0)
df['finished'] = df['pos_numeric'].notna().astype(int)

In [17]:
# Converts weight format to pounds
parts = df["wgt"].str.split("-", expand=True)

df["wgt_lbs"] = (parts[0].astype(float) * 14 + parts[1].astype(float))

In [18]:
df['dam_clean'] = df['dam'].str.replace(r'[()]', '', regex=True).str.strip()

In [19]:
df['horse_clean'] = df['horse'].str.replace(r'[()]', '', regex=True).str.strip()

In [20]:
# Estimate birth year from race age
df["birth_year"] = df["date"].dt.year - df["age"]

# Convert to current age
current_year = pd.Timestamp.today().year
df["current_age"] = current_year - df["birth_year"]

In [21]:
df['dam_clean'] = (
    df['dam_clean']
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [22]:
df['horse_clean'] = (
    df['horse_clean']
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [23]:
df['sire_clean'] = (
    df['sire']
    .str.replace(r'[()]', '', regex=True).str.strip()
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [24]:
df['damsire_clean'] = (
    df['damsire']
    .str.replace(r'[()]', '', regex=True).str.strip()
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [26]:
horse_mapping = {
    "angea cara fr": "angela cara fr",
    "hardi blue trois aa fr": "hardi blue trois fr",
    "sublime chope fr": "sublime chop fr"
}
df["horse_final"] = df["horse_clean"].replace(horse_mapping)

In [27]:
damsire_mapping = {
    "a p indy": "ap indy",
    "ut*windsor heights" : "windsor heights",
    "ut*mangarose" : "mangarose",
    "almutawakel" : "almutawakeli"
}
mask = (
    df["dam_clean"].eq("private jet usa")
    & df["damsire_clean"].isna()
)

df.loc[mask, "damsire_clean"] = "smart strike"
df["damsire_clean"] = df["damsire_clean"].str.rstrip()

df["damsire_final"] = df["damsire_clean"].replace(horse_mapping)

In [28]:
dam_mapping = {
    "o k angie arg" : "ok angie arg",
    "moonlight shadow gb" : "moon light shadow gb",
    "sound out ire" : "soundout ire",
    "ticker tape gb" : "ticker tapei gb",
    "ascolini aus" : "ascolini nz",
    "nation ii usa" : "nation usa",
    "sun song ii fr" : "sun song fr"
}
mask = (
    df["horse_clean"].eq("hapi jpn")
    & df["damsire_clean"].isna()
)

df.loc[mask, "dam_clean"] = "queen pirates jpn"

df["dam_final"] = df["dam_clean"].replace(horse_mapping)

In [29]:
df["dam_id"] = (
    df["dam_final"].str.strip().str.replace(r"\s+", "_", regex=True)
    + "_" +
    df["damsire_clean"].fillna("unknown_dam").str.strip().str.replace(r"\s+", "_", regex=True)
)

In [30]:
df["horse_id"] = (
    df["horse_final"].str.strip().str.replace(r"\s+", "_", regex=True)
    + "_" +
    df["dam_id"].fillna("unknown_dam").str.strip().str.replace(r"\s+", "_", regex=True)
)

In [31]:
df["damsire_id"] = (
    df["damsire_final"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [32]:
df["sire_id"] = (
    df["sire_clean"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [33]:
def clean_names(name):
    if pd.isna(name):
        return None

    name = name.lower().strip()
    name = re.sub(r"\b(mr|mrs|ms|miss|dr)\b", "", name)
    name = name.replace("'", "")
    name = re.sub(r"[^a-z\s]", " ", name)
    name = " ".join(name.split())

    return name



In [34]:
df["jockey_clean"] = df["jockey"].apply(clean_names)

In [35]:
df["trainer_clean"] = df["trainer"].apply(clean_names)

In [36]:
df["owner_clean"] = df["owner"].apply(clean_names)

In [37]:
jockey_mapping = {
    "E Dwan": "Evan Dwan",
    "M J M OSullivan": "Michael OSullivan"
}
df["jockey_clean"] = df["jockey_clean"].replace(jockey_mapping)

In [38]:
# Names that must keep their titles as separate identities
keep_titles = {
    "mr a jones",
    "miss a jones",
    "miss m osullivan",
    "ms m osullivan"
}


# Restore original titled names for these exceptions
df.loc[
    df["jockey"].str.lower().isin(keep_titles),
    "jockey_clean"
] = df.loc[
    df["jockey"].str.lower().isin(keep_titles),
    "jockey"
].str.lower()

In [39]:
df["jockey_id"] = (
    df["jockey_clean"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [40]:
# -----------------------------------
# Remove titles from names
# -----------------------------------

def strip_titles(name):

    titles = {"mr", "mrs", "ms", "miss"}

    return [
        x.lower()
        for x in name.split()
        if x.lower() not in titles
    ]



# -----------------------------------
# Match two cleaned names
# -----------------------------------

def matches_name(base_parts, candidate_parts):

    # surname
    if base_parts[-1] != candidate_parts[-1]:
        return False


    # first name
    base_first = base_parts[0]
    candidate_first = candidate_parts[0]


    if len(base_first) == 1:

        if candidate_first[0] != base_first:
            return False

    else:

        if candidate_first != base_first:
            return False



    # middle names

    base_middle = base_parts[1:-1]
    candidate_middle = candidate_parts[1:-1]


    for i, middle in enumerate(base_middle):

        if i >= len(candidate_middle):
            return False


        if len(middle) == 1:

            if candidate_middle[i][0] != middle:
                return False

        else:

            if candidate_middle[i] != middle:
                return False


    return True



# -----------------------------------
# Check whether candidates are
# genuinely conflicting identities
# -----------------------------------

def candidate_group_conflict(candidates):

    cleaned = [
        strip_titles(name)
        for name in candidates
    ]


    for i in range(len(cleaned)):

        for j in range(i + 1, len(cleaned)):

            a = cleaned[i]
            b = cleaned[j]


            # surname
            if a[-1] != b[-1]:
                return True


            # first name
            if (
                len(a[0]) > 1
                and len(b[0]) > 1
                and a[0] != b[0]
            ):
                return True


            # middle names

            middle_a = a[1:-1]
            middle_b = b[1:-1]


            if middle_a and middle_b:

                if len(middle_a) != len(middle_b):
                    return True


                for x, y in zip(middle_a, middle_b):

                    if len(x) > 1 and len(y) > 1:

                        if x != y:
                            return True


                    elif len(x) == 1 and len(y) > 1:

                        if x != y[0]:
                            return True


                    elif len(y) == 1 and len(x) > 1:

                        if y != x[0]:
                            return True


    return False

def match_table(entity):

  entity_clean = f"{entity}_clean"


  # -----------------------------------
  # Store original names for display
  # -----------------------------------

  entity_display = (
      df[[entity_clean, entity]]
      .dropna()
      .drop_duplicates()
      .groupby(entity_clean)[entity]
      .apply(list)
      .to_dict()
  )



  # -----------------------------------
  # Unique cleaned names
  # -----------------------------------

  entity_names = pd.DataFrame({
      entity_clean: df[entity_clean]
      .dropna()
      .unique()
  })


  entity_names["parts"] = (
      entity_names[entity_clean]
      .str.split()
  )


  entity_names = entity_names[
      entity_names["parts"].str.len() >= 2
  ].copy()


  entity_names["surname"] = (
      entity_names["parts"]
      .str[-1]
  )



  # -----------------------------------
  # Group by surname
  # -----------------------------------

  surname_groups = defaultdict(list)


  for row in entity_names.itertuples(index=False):

      surname_groups[row.surname].append(
          (
              getattr(row, entity_clean),
              row.parts
          )
      )

      # -----------------------------------
  # Find possible matches
  # -----------------------------------

  results = []


  for row in entity_names.itertuples(index=False):

      base_name = getattr(row, entity_clean)
      base_parts = row.parts
      surname = row.surname


      candidates = []


      for candidate_name, candidate_parts in surname_groups[surname]:

          if candidate_name == base_name:
              continue

          if matches_name(base_parts, candidate_parts):
              candidates.append(candidate_name)



      candidates = sorted(set(candidates))



      # -----------------------------------
      # Keep only genuine conflicts
      # -----------------------------------

      if (
          len(candidates) >= 2
          and candidate_group_conflict(candidates)
      ):


          base_display = entity_display.get(
              base_name,
              [base_name]
          )


          candidate_display = []

          for candidate in candidates:

              candidate_display.extend(
                  entity_display.get(
                      candidate,
                      [candidate]
                  )
              )

          candidate_display = sorted(
              set(candidate_display)
          )

          results.append({
              "base_name": ", ".join(base_display),
              "n_candidates": len(candidate_display),
              "candidates": ", ".join(candidate_display)

          })



  # -----------------------------------
  # Initial ambiguity table
  # -----------------------------------

  final_match_table = pd.DataFrame(results)
  return final_match_table

In [41]:
def overlap(entity, final_match_table):

  entity_clean = f"{entity}_clean"

  entity_original_to_clean = (
      df[[entity, entity_clean]]
      .dropna()
      .drop_duplicates()
      .set_index(entity)[entity_clean]
      .to_dict()
  )


  # -----------------------------------
  # entity_clean -> horse_clean set
  # -----------------------------------

  entity_horses = (
      df.dropna(subset=[entity_clean, "horse_id"])
        .groupby(entity_clean)["horse_id"]
        .apply(set)
        .to_dict()
  )


  # -----------------------------------
  # entity name -> horses
  # -----------------------------------

  def entity_to_horses(name):

      clean_name = entity_original_to_clean.get(name)

      if clean_name is None:
          return set()

      return entity_horses.get(clean_name, set())



  # -----------------------------------
  # Find overlaps
  # -----------------------------------

  def find_overlap_pairs(row):

      base = row["base_name"]
      candidates = row["candidates"].split(", ")

      base_candidate_pairs = []
      candidate_candidate_pairs = []


      # -------------------------------
      # Base vs candidate
      # -------------------------------

      for candidate in candidates:

          overlap = (
              entity_to_horses(base)
              .intersection(
                  entity_to_horses(candidate)
              )
          )

          if overlap:
              base_candidate_pairs.append(
                  f"{base} - {candidate}"
              )


      # -------------------------------
      # Candidate vs candidate
      # -------------------------------

      for a, b in combinations(candidates, 2):

          overlap = (
              entity_to_horses(a)
              .intersection(
                  entity_to_horses(b)
              )
          )

          if overlap:
              candidate_candidate_pairs.append(
                  f"{a} - {b}"
              )


      return pd.Series({
          "base_candidate_overlap_pairs": "; ".join(base_candidate_pairs),
          "candidate_candidate_overlap_pairs": "; ".join(candidate_candidate_pairs)
      })



  # -----------------------------------
  # Add columns
  # -----------------------------------

  entity_overlap_table = (
      final_match_table
      .copy()
  )


  entity_overlap_table[
      [
          "base_candidate_overlap_pairs",
          "candidate_candidate_overlap_pairs"
      ]
  ] = (
      entity_overlap_table
      .apply(find_overlap_pairs, axis=1)
  )

  return entity_overlap_table

In [42]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.9Gi       392Mi       3.0Mi       7.4Gi       7.4Gi
Swap:             0B          0B          0B


In [43]:
def map_overlap(entity, candidate_table):

  entity_overlap_table = candidate_table.copy()

  G = nx.Graph()

  for _, row in entity_overlap_table.iterrows():

      pairs = []

      if row["base_candidate_overlap_pairs"]:
          pairs.extend(
              row["base_candidate_overlap_pairs"].split("; ")
          )

      if row["candidate_candidate_overlap_pairs"]:
          pairs.extend(
              row["candidate_candidate_overlap_pairs"].split("; ")
          )


      for pair in pairs:

          entity1, entity2 = pair.split(" - ")

          G.add_edge(entity1, entity2)



  # -----------------------------------
  # Count original entity appearances
  # -----------------------------------

  entity_counts = (
      df[entity]
      .value_counts()
      .to_dict()
  )


  # -----------------------------------
  # Map each connected group to
  # most common entity name
  # -----------------------------------

  entity_mapping = {}

  for component in nx.connected_components(G):

      canonical_name = max(
          component,
          key=lambda x: entity_counts.get(x, 0)
      )

      for entity in component:
          entity_mapping[entity] = canonical_name

  return entity_mapping

In [44]:
trainer_table = match_table("trainer")
trainer_overlap = overlap("trainer", trainer_table)
trainer_map = map_overlap("trainer", trainer_overlap)
df["trainer_id"] = (
    df["trainer"]
    .replace(trainer_map)
    .str.lower()
    .str.replace(" ", "_")
)

In [45]:
owner_table = match_table("owner")
owner_overlap = overlap("owner", owner_table)
owner_map = map_overlap("owner", owner_overlap)
df["owner_id"] = (
    df["owner"]
    .replace(trainer_map)
    .str.lower()
    .str.replace(" ", "_")
)

In [65]:
#create form
df = df.sort_values(['horse_id', 'date'])

df['horse_ewa_pos'] = (
    df.groupby('horse_id')['pos_numeric']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

df['horse_ewa_btn'] = (
    df.groupby('horse_id')['ovr_btn']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['horse_ewa_finish_rate'] = (
    df.groupby('horse_id')['finished']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

In [84]:
df = df.sort_values(['jockey_id', 'date'])

df['jockey_ewa_pos'] = (
    df.groupby('jockey_id')['pos_numeric']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['jockey_ewa_btn'] = (
    df.groupby('jockey_id')['ovr_btn']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['jockey_ewa_finish_rate'] = (
    df.groupby('jockey_id')['finished']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

In [67]:
df['horse_jockey_ewa_pos'] = (
    df.groupby(['horse_id', 'jockey_id'])['pos_numeric']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)
df['horse_jockey_ewa_btn'] = (
    df.groupby(['horse_id', 'jockey_id'])['ovr_btn']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)
df['horse_jockey_ewa_finish_rate'] = (
    df.groupby(['horse_id', 'jockey_id'])['finished']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)

In [68]:
#target
df["top3"] = (df["pos_numeric"] <= 3).astype(float)

In [85]:
ignore_cols = ['horse', 'dam', 'sire', 'damsire', 'jockey', 'trainer', 'owner', 'pos', 'date', 'wgt', 'jockey_clean', 'trainer_clean', 'owner_clean', 'horse_clean', 'dam_clean', 'damsire_clean', 'sire_clean','dam_final', 'horse_final', 'damsire_final' , 'draw', 'off', 'or', 'pos_numeric', 'ovr_btn']
target_col = ['top3']
cat_cols = ['horse_id', 'dam_id', 'sire_id', 'damsire_id', 'jockey_id', 'trainer_id', 'owner_id', 'course', 'race_class', 'month', 'race_type', 'sex', 'going', 'age_band',]
num_cols = [c for c in df.columns if c not in cat_cols + ignore_cols + target_col]
#clean up the horse_clean / horse_final stuff and draw, off, or, pos_numeric, ovr_btn

In [70]:
#loss
y = df[target_col].copy()
d = pd.to_numeric(df["ovr_btn"], errors="coerce").fillna(0)
w = torch.where(
    torch.tensor(y.values.squeeze(), dtype=torch.float32) == 1,
    torch.tensor(1.0),
    1 - torch.exp(-0.3 * torch.tensor(d.values, dtype=torch.float32))
)
def weighted_BCE_loss(p, y, w):
    loss = - (w * y * torch.log(p) + w * (1 - y) * torch.log(1 - p))
    return loss.mean()

In [86]:
# Sort chronologically first
df = df.sort_values("date").reset_index(drop=True)

# Create feature sets
X_cat = df[cat_cols].copy()
X_num = df[num_cols].copy()

n = len(df)

#chronological split
train_end = int(n * 0.70)
val_end = int(n * 0.90)

w_train = w[:train_end]
w_val = w[train_end:val_end]
w_test = w[val_end:]

# Train
X_cat_train = X_cat.iloc[:train_end].copy()
X_num_train = X_num.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()


# Validation
X_cat_val = X_cat.iloc[train_end:val_end].copy()
X_num_val = X_num.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()


# Test
X_cat_test = X_cat.iloc[val_end:].copy()
X_num_test = X_num.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

X_num_train = X_num_train.fillna(0)
X_num_val = X_num_val.fillna(0)
X_num_test = X_num_test.fillna(0)

scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

cat_maps = {}

for col in cat_cols:

    # Fit mapping only on training data
    labels, uniques = pd.factorize(
        X_cat_train[col]
    )
    cat_maps[col] = uniques

    mapping = {
        value: idx
        for idx, value in enumerate(uniques)
    }

    X_cat_train[col] = labels
    X_cat_val[col] = X_cat_val[col].map(mapping)
    X_cat_test[col] = X_cat_test[col].map(mapping)
    X_cat_val[col] = X_cat_val[col].fillna(0)
    X_cat_test[col] = X_cat_test[col].fillna(0)

X_cat_train = torch.tensor(
    X_cat_train.values,
    dtype=torch.long
)

X_cat_val = torch.tensor(
    X_cat_val.values,
    dtype=torch.long
)

X_cat_test = torch.tensor(
    X_cat_test.values,
    dtype=torch.long
)


X_num_train = torch.tensor(
    X_num_train,
    dtype=torch.float32
)

X_num_val = torch.tensor(
    X_num_val,
    dtype=torch.float32
)

X_num_test = torch.tensor(
    X_num_test,
    dtype=torch.float32
)


y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

y_val = torch.tensor(
    y_val.values,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
)

In [87]:
class RacingDataset(Dataset):
    def __init__(self, X_cat, X_num, y, w):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = y
        self.w = w

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_cat[idx],
            self.X_num[idx],
            self.y[idx],
            self.w[idx]
        )

In [88]:
train_dataset = RacingDataset(
    X_cat_train,
    X_num_train,
    y_train,
    w_train
)

val_dataset = RacingDataset(
    X_cat_val,
    X_num_val,
    y_val,
    w_val
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [74]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       7.7Gi       1.5Gi       3.0Mi       3.5Gi       4.7Gi
Swap:             0B          0B          0B


In [90]:
#create runner_id
df = df.reset_index(drop=True)
df["runner_id"] = df.index

In [ ]:
race_num_cols = ["year", "dist"]
horse_num_cols = ["sex_idx"]
runner_num_cols = ["wgt_lbs", "horse_jockey_ewa_pos", "horse_jockey_ewa_btn", "horse_jockey_ewa_finish_rate", "jockey_ewa_pos", "jockey_ewa_btn", "jockey_ewa_finish_rate", "horse_ewa_pos", "horse_ewa_btn", "horse_ewa_finish_rate"]
#what to do about currently ignored features

In [ ]:
race_df = pd.get_dummies(
    df[["race_id"] + race_num_cols + ["going", "age_band", "race_type", "month", "class", "course"]]
    .drop_duplicates("race_id"),
    columns=["going", "age_band", "race_type", "month", "class"],
    dtype=float
)
horse_df = df[["horse_id"] + horse_num_cols+ ["sex", "dam_id", "damsire_id", "sire_id"]].drop_duplicates("horse_id")
runner_df = df[["runner_id"] + runner_num_cols].drop_duplicates("runner_id")
jockey_df = df[["jockey_id"]].drop_duplicates("jockey_id")
trainer_df = df[["trainer_id"]].drop_duplicates("trainer_id")
owner_df = df[["owner_id"]].drop_duplicates("owner_id")
dam_df = df[["dam_id"]].drop_duplicates("jockey_id")
trainer_df = df[["trainer_id"]].drop_duplicates("trainer_id")
owner_df = df[["owner_id"]].drop_duplicates("owner_id")

In [ ]:
#mappings for categorical features that will be embedded
sex_map = {
    "male": 0,
    "female": 1
}
course_map = {
    v: i for i, v in enumerate(df["race_course"].dropna().unique())
}
dam_map = {
    v: i for i, v in enumerate(df["dam_id"].dropna().unique())
}
sire_map = {
    v: i for i, v in enumerate(df["sire_id"].dropna().unique())
}
damsire_map = {
    v: i for i, v in enumerate(df["damsire_id"].dropna().unique())
}
race_df["course_idx"] = race_df["race_course"].map(course_map).fillna(-1).astype(int)
horse_df["dam_idx"] = race_df["dams_id"].map(dam_map).fillna(-1).astype(int)
horse_df["sire_idx"] = race_df["sire_id"].map(sire_map).fillna(-1).astype(int)
horse_df["damsire_idx"] = race_df["damsire_id"].map(damsire_map).fillna(-1).astype(int)
horse_df["sex_idx"] = horse_df["sex"].map(sex_map)
#what to do about currently ignored features

In [ ]:
#exclude ids, categories and category indexes
race_x_cols = [
    c for c in race_df.columns
    if c not in ["race_id", "course", "course_idx"]
]
horse_x_cols = [
    c for c in horse_df.columns
    if c not in ["horse_id", "sex", "dam_id", "sire_id", "damsire_id"]
]

In [ ]:
#numerical columns to torch tensors
race_x = torch.tensor(
    race_df[race_x_cols].values,
    dtype=torch.float32
)
horse_x = torch.tensor(
    horse_df[horse_x_cols].values,
    dtype=torch.float32
)
runner_x = torch.tensor(
    runner_df[runner_num_cols].values,
    dtype=torch.float32
)
jockey_x = torch.ones((len(jockey_df), 1))
trainer_x = torch.ones((len(trainer_df), 1))
owner_x = torch.ones((len(owner_df), 1))

In [ ]:
#converting to pytorch tensors
race_course = torch.tensor(
    race_df["course_idx"].values,
    dtype=torch.long
)
horse_dam = torch.tensor(
    horse_df["dam_idx"].values,
    dtype=torch.long
)
horse_sire = torch.tensor(
    horse_df["sire_idx"].values,
    dtype=torch.long
)
horse_damsire = torch.tensor(
    horse_df["damsire_idx"].values,
    dtype=torch.long
)

In [ ]:
#node mappings
race_map = {
    race_id: i
    for i, race_id in enumerate(race_df["race_id"])
}

horse_map = {
    horse_id: i
    for i, horse_id in enumerate(horse_df["horse_id"])
}

runner_map = {
    runner_id: i
    for i, runner_id in enumerate(runner_df["runner_id"])
}

jockey_map = {
    jockey_id: i
    for i, jockey_id in enumerate(jockey_df["jockey_id"])
}

trainer_map = {
    trainer_id: i
    for i, trainer_id in enumerate(trainer_df["trainer_id"])
}

owner_map = {
    owner_id: i
    for i, owner_id in enumerate(owner_df["owner_id"])
}

In [ ]:
#edges
runner_to_jockey = torch.tensor([
    [runner_map[r], jockey_map[j]]
    for r, j in zip(df["runner_id"], df["jockey_id"])
], dtype=torch.long).T

runner_to_trainer = torch.tensor([
    [runner_map[r], trainer_map[t]]
    for r, t in zip(df["runner_id"], df["trainer_id"])
], dtype=torch.long).T

runner_to_owner = torch.tensor([
    [runner_map[r], owner_map[o]]
    for r, o in zip(df["runner_id"], df["owner_id"])
], dtype=torch.long).T

runner_to_race = torch.tensor([
    [runner_map[r], race_map[ra]]
    for r, ra in zip(df["runner_id"], df["race_id"])
], dtype=torch.long).T

runner_to_horse = torch.tensor([
    [runner_map[r], horse_map[ra]]
    for r, ra in zip(df["runner_id"], df["horse_id"])
], dtype=torch.long).T

In [ ]:
data = HeteroData()
data["race"].x = race_x
data["race"].course_idx = race_course
data["runner"].x = runner_x
data["horse"].x = horse_x
data["horse"].dam_idx = horse_dam
data["horse"].sire_idx = horse_sire
data["horse"].damsire_idx = horse_damsire
data["jockey"].x = jockey_x
data["trainer"].x = trainer_x
data["owner"].x = owner_x

#edges go both ways
data["runner", "runner_to_race", "race"].edge_index = runner_to_race
data["runner", "runner_to_horse", "horse"].edge_index = runner_to_horse
data["runner", "runner_to_jockey", "jockey"].edge_index = runner_to_jockey
data["runner", "runner_to_trainer", "trainer"].edge_index = runner_to_trainer
data["runner", "runner_to_owner", "owner"].edge_index = runner_to_owner
data["race", "race_to_runner", "runner"].edge_index = runner_to_race
data["horse", "horse_to_runner", "runner"].edge_index = runner_to_horse
data["jockey", "jockey_to_runner", "runner"].edge_index = runner_to_jockey
data["trainer", "trainer_to_runner", "runner"].edge_index = runner_to_trainer
data["owner", "owner_to_runner", "runner"].edge_index = runner_to_owner

# Add reverse edges automatically
data = ToUndirected()(data)

In [ ]:
!free -h

In [ ]:
n_horse_features = horse_x.shape[1]
n_race_features = race_x.shape[1]
n_runner_features = runner_x.shape[1]
n_jockey_features = jockey_x.shape[1]
n_trainer_features = trainer_x.shape[1]
n_owner_features = owner_x.shape[1]

In [ ]:
#based off MLP format
class HorsePredictionGNN(nn.model):
  def __innit__(self,n_horse_features, n_race_features, n_runner_features, n_jockey_features, n_trainer_features, n_owner_features, dropout= 0.2, ):
    super().__innit__()
    self.course_emb = nn.Embedding(len(course_map), 8)
    self.dam_emb = nn.Embedding(len(dam_map), 8)
    self.sire_emb = nn.Embedding(len(sire_map), 8)
    self.damsire_emb = nn.Embedding(len(damsire_map), 8)

    self.horse_lin = nn.Sequential(
          nn.Linear(n_horse_features + 24, 512),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(512, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
        )
    self.race_lin = nn.Sequential(
          nn.Linear(n_race_features + 8, 512),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(512, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
        )
    self.runner_lin = nn.Sequential(
          nn.Linear(n_runner_features, 512),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(512, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
        )
    self.jockey_lin = nn.Sequential(
          nn.Linear(n_jockey_features, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
              )
    self.trainer_lin = nn.Sequential(
          nn.Linear(n_jockey_features, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
              )
    self.runner_lin = nn.Sequential(
          nn.Linear(n_runner_features, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
        )

    self.conv1 = SAGEConv((-1, -1), 128)
    self.conv2 = SAGEConv((-1, -1), 128)

    self.predictor = nn.Sequential(
    nn.ReLU(),
    nn.Linear(128, output_size),
    nn.Sigmoid()
    )

  def forward(self, data):
    course_emb = self.course_embedding(data["race"].course_idx)

In [ ]:
#code taken from github
class HeteroGNN(torch.nn.Module):
    def __init__(self, metadata, hidden_channels, out_channels, num_layers):
        super().__init__()

        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((-1, -1), hidden_channels)
                for edge_type in metadata[1]
            })
            self.convs.append(conv)

        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: F.leaky_relu(x) for key, x in x_dict.items()}
        return self.lin(x_dict['author'])


model = HeteroGNN(data.metadata(), hidden_channels=128, out_channels=1,
                  num_layers=2)